<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/02-data-structures/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2: Exercises

Twelve problems: four Easy, five Medium and three Hard. As in Module 1, each one is a function to fill in, followed by a check cell.

**Run the setup cell first.**

Rules for this module:

- You can import from the standard library, especially `collections` and `heapq`. That's the point of this module. No NumPy or pandas yet.
- Think about which data structure fits each problem before you start writing. Several of these have a one-line answer if you pick the right one.
- Don't modify the inputs. The checks will tell you if you do.
- Solutions are in `03_solutions.ipynb`.

In [ ]:
# Setup: run this cell once before you start.
# It defines the check functions used below. You don't need to read it,
# but feel free to look at the test cases if you're stuck.

import copy
import math


class Raises:
    """Marks a test case where the function is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, float) or isinstance(b, float):
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=60):
    """repr() that stays readable for long lists."""
    text = repr(x)
    if len(text) <= limit:
        return text
    if isinstance(x, tuple):
        return "(" + ", ".join(_short(item, limit) for item in x) + ")"
    if isinstance(x, list):
        head = ", ".join(repr(item) for item in x[:3])
        return f"[{head}, ... {len(x)} items]"
    return text[:limit - 3] + "..."


def _call_str(func, args):
    return f"{func.__name__}({', '.join(_short(a) for a in args)})"


def _run(func, cases):
    for args, expected, hint in cases:
        call = _call_str(func, args)
        before = copy.deepcopy(args)
        try:
            result = func(*args)
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {call}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {expected!r}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {call}")
            print(f"      expected {expected!r}, but it returned {result!r}")
            if hint:
                print(f"      hint: {hint}")
            return
        if args != before:
            print(f"FAIL  {call}")
            print("      your function modified its input. It should leave it unchanged.")
            return
        if not _same(result, expected):
            print(f"FAIL  {call}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {func.__name__}")


def check_unique_in_order(f):
    _run(f, [
        (([3, 1, 3, 2, 1],), [3, 1, 2], "keep the order of first appearance. A set doesn't remember order."),
        ((["b", "a", "b"],), ["b", "a"], None),
        (([],), [], None),
        (([1, 1, 1],), [1], None),
    ])


def check_invert_dict(f):
    _run(f, [
        (({"cat": 0, "dog": 1},), {0: "cat", 1: "dog"}, None),
        (({},), {}, None),
        (({"a": 1, "b": 1},), Raises(ValueError), "two keys share the value 1, so the inverted dict would lose one of them"),
    ])


def check_merge_configs(f):
    _run(f, [
        (({"lr": 0.001, "epochs": 10}, {"lr": 0.01}), {"lr": 0.01, "epochs": 10}, None),
        (({"a": 1}, {}), {"a": 1}, None),
        (({}, {"b": 2}), {"b": 2}, None),
        (({"a": 1, "b": 2}, {"b": 3, "c": 4}), {"a": 1, "b": 3, "c": 4}, None),
    ])


def check_oov_words(f):
    _run(f, [
        ((["the", "cat", "sat"], ["the", "dog", "sat", "dog"]), ["dog"], "each word should appear once"),
        ((["a", "b"], ["c", "b", "a", "d"]), ["c", "d"], None),
        ((["x"], ["x", "x"]), [], None),
        (([], ["b", "a"]), ["a", "b"], "return the words sorted"),
    ])


def check_group_by_label(f):
    _run(f, [
        (([([1, 2], "cat"), ([3, 4], "dog"), ([5, 6], "cat")],),
         {"cat": [[1, 2], [5, 6]], "dog": [[3, 4]]}, None),
        (([],), {}, None),
        (([("a", 1), ("b", 1), ("c", 2)],), {1: ["a", "b"], 2: ["c"]}, None),
    ])


def check_top_k(f):
    scores = {"cat": 0.6, "dog": 0.3, "fox": 0.1}
    _run(f, [
        ((scores, 2), [("cat", 0.6), ("dog", 0.3)], None),
        ((scores, 5), [("cat", 0.6), ("dog", 0.3), ("fox", 0.1)], "if k is bigger than the number of classes, return all of them"),
        (({"b": 0.5, "a": 0.5, "c": 0.2}, 2), [("a", 0.5), ("b", 0.5)], "break ties alphabetically by label"),
        (({"x": 0.9}, 0), [], None),
    ])


def check_word_frequencies(f):
    _run(f, [
        (("The cat saw the other cat.",), {"the": 2, "cat": 2, "saw": 1, "other": 1}, "lowercase everything and strip punctuation from the ends of words"),
        (("Free! Free money, free.", 2), {"free": 3}, None),
        (("",), {}, None),
        (("Hello , world",), {"hello": 1, "world": 1}, "a lone ',' becomes an empty string after stripping. Skip it."),
    ])


def check_deep_get(f):
    _run(f, [
        (({"model": {"lr": 0.01}}, "model.lr"), 0.01, None),
        (({"model": {"lr": 0.01}}, "model.momentum"), None, None),
        (({"model": {"lr": 0.01}}, "optimizer.lr", 0.001), 0.001, None),
        (({"a": {"b": {"c": 5}}}, "a.b.c"), 5, None),
        (({"a": {"b": 3}}, "a.b.c", "missing"), "missing", "config['a']['b'] is 3, not a dict, so there's nothing to look inside"),
        (({"lr": 0.0}, "lr", 1.0), 0.0, "0.0 is a real value, not a missing one"),
        (({"a": {"b": None}}, "a.b", 7), None, "the key exists and its value is None, so return None, not the default"),
    ])


def check_records_to_columns(f):
    _run(f, [
        (([{"a": 1, "b": 2}, {"a": 3, "b": 4}],), {"a": [1, 3], "b": [2, 4]}, None),
        (([{"a": 1}, {"b": 2}],), {"a": [1, None], "b": [None, 2]}, "fill missing values with None"),
        (([],), {}, None),
        (([{"x": "p", "y": 0.5}],), {"x": ["p"], "y": [0.5]}, None),
    ])


def check_stratified_split(f):
    s1 = [(1, "a"), (2, "b"), (3, "a"), (4, "a"), (5, "b"), (6, "a")]
    s3 = [(i, "x") for i in range(10)]
    s4 = [(i, "spam" if i in (2, 7) else "ham") for i in range(10)]
    _run(f, [
        ((s1, 0.5), ([(1, "a"), (2, "b"), (3, "a")], [(4, "a"), (5, "b"), (6, "a")]), None),
        ((s1, 0.0), (s1, []), "with test_ratio=0 nothing goes to test. Careful: idx[-0:] is the whole list"),
        ((s3, 0.2), (s3[:8], s3[8:]), None),
        ((s4, 0.5),
         ([(0, "ham"), (1, "ham"), (2, "spam"), (3, "ham"), (4, "ham")],
          [(5, "ham"), (6, "ham"), (7, "spam"), (8, "ham"), (9, "ham")]),
         "split each label separately, then keep both outputs in the original order"),
    ])


def check_most_similar(f):
    docs = {"d1": {"cat", "sat", "mat"}, "d2": {"dog", "sat", "log"},
            "d3": {"cat", "mat", "hat"}, "d4": set()}
    _run(f, [
        (({"cat", "mat"}, docs, 2), ["d1", "d3"], "d1 and d3 tie, so sort them by name"),
        (({"dog", "log"}, docs, 1), ["d2"], None),
        (({"cat", "mat"}, docs, 10), ["d1", "d3", "d2", "d4"], None),
        ((set(), {"a": set(), "b": {"x"}}, 2), ["a", "b"], "two empty sets have an empty union. Treat their similarity as 0.0"),
    ])


def check_bigram_model(f):
    _run(f, [
        ((["the cat sat", "the cat ran"],), {"the": {"cat": 2}, "cat": {"sat": 1, "ran": 1}}, "count pairs inside each sentence only. 'sat' is not followed by 'the'"),
        ((["a b a b"],), {"a": {"b": 2}, "b": {"a": 1}}, None),
        ((["hello"],), {}, "a single word has no next word"),
        ((["I am", "am I"],), {"i": {"am": 1}, "am": {"i": 1}}, "lowercase the words, and don't count pairs across two sentences"),
    ])


print("Setup done. Checks are ready.")

## Easy

### Exercise 1: `unique_in_order`

Write `unique_in_order(items)` that removes duplicates from a list but keeps the order in which items first appear.

```python
unique_in_order([3, 1, 3, 2, 1])   # [3, 1, 2]
```

In [ ]:
def unique_in_order(items):
    # your code here
    pass

In [ ]:
check_unique_in_order(unique_in_order)

### Exercise 2: `invert_dict`

Write `invert_dict(d)` that swaps keys and values.

```python
invert_dict({"cat": 0, "dog": 1})   # {0: "cat", 1: "dog"}
```

If two keys have the same value, the inverted dictionary can't hold both, so raise a `ValueError` instead.

This is how you get from a `label_to_id` map to an `id_to_label` map.

In [ ]:
def invert_dict(d):
    # your code here
    pass

In [ ]:
check_invert_dict(invert_dict)

### Exercise 3: `merge_configs`

Write `merge_configs(defaults, overrides)` that returns a new dictionary with all the keys from both. When a key is in both, the value from `overrides` wins.

```python
merge_configs({"lr": 0.001, "epochs": 10}, {"lr": 0.01})
# {"lr": 0.01, "epochs": 10}
```

Don't modify either of the input dictionaries. Only the top level needs merging (no nested dicts).

In [ ]:
def merge_configs(defaults, overrides):
    # your code here
    pass

In [ ]:
check_merge_configs(merge_configs)

### Exercise 4: `oov_words`

Write `oov_words(train_tokens, test_tokens)` that returns the words in `test_tokens` that never appear in `train_tokens`. Each word should appear once, and the result should be sorted.

```python
oov_words(["the", "cat", "sat"], ["the", "dog", "sat", "dog"])   # ["dog"]
```

These are the out-of-vocabulary (OOV) words: words the model will meet at test time but never saw in training.

In [ ]:
def oov_words(train_tokens, test_tokens):
    # your code here
    pass

In [ ]:
check_oov_words(oov_words)

## Medium

### Exercise 5: `group_by_label`

Write `group_by_label(samples)` that takes a list of `(features, label)` pairs and returns a dictionary mapping each label to a list of the features with that label, in their original order.

```python
group_by_label([([1, 2], "cat"), ([3, 4], "dog"), ([5, 6], "cat")])
# {"cat": [[1, 2], [5, 6]], "dog": [[3, 4]]}
```

In [ ]:
def group_by_label(samples):
    # your code here
    pass

In [ ]:
check_group_by_label(group_by_label)

### Exercise 6: `top_k`

Write `top_k(scores, k)` where `scores` maps class names to scores. Return a list of the `k` highest `(label, score)` pairs, highest score first. If two classes have the same score, put them in alphabetical order. If `k` is larger than the number of classes, return all of them.

```python
top_k({"cat": 0.6, "dog": 0.3, "fox": 0.1}, 2)
# [("cat", 0.6), ("dog", 0.3)]
```

In [ ]:
def top_k(scores, k):
    # your code here
    pass

In [ ]:
check_top_k(top_k)

### Exercise 7: `word_frequencies`

Write `word_frequencies(text, min_count=1)` that counts words in a piece of text:

- Lowercase the text and split it on whitespace.
- Strip these punctuation characters from the start and end of each word: `.,!?;:`
- Skip anything that becomes an empty string.
- Only include words that appear at least `min_count` times.

Return a dictionary mapping word to count.

```python
word_frequencies("The cat saw the other cat.")
# {"the": 2, "cat": 2, "saw": 1, "other": 1}
```

You may use `Counter`.

In [ ]:
def word_frequencies(text, min_count=1):
    # your code here
    pass

In [ ]:
check_word_frequencies(word_frequencies)

### Exercise 8: `deep_get`

Write `deep_get(config, path, default=None)` that looks up a value in nested dictionaries using a dotted path.

```python
config = {"model": {"optimizer": {"lr": 0.01}}}
deep_get(config, "model.optimizer.lr")          # 0.01
deep_get(config, "model.scheduler.step", 10)    # 10
```

Return `default` if any key along the path is missing, or if you hit something that isn't a dictionary before the path ends. If a key exists, return its value even if that value is `0`, `False` or `None`.

In [ ]:
def deep_get(config, path, default=None):
    # your code here
    pass

In [ ]:
check_deep_get(deep_get)

### Exercise 9: `records_to_columns`

Data often comes as a list of records (one dict per row), but many tools want it as columns (one list per field). Write `records_to_columns(records)` that converts from one to the other.

```python
records_to_columns([{"a": 1, "b": 2}, {"a": 3, "b": 4}])
# {"a": [1, 3], "b": [2, 4]}
```

If a record is missing a key that other records have, use `None` for that value.

```python
records_to_columns([{"a": 1}, {"b": 2}])
# {"a": [1, None], "b": [None, 2]}
```

In [ ]:
def records_to_columns(records):
    # your code here
    pass

In [ ]:
check_records_to_columns(records_to_columns)

## Hard

### Exercise 10: `stratified_split`

In Module 1 you split data by taking the last part as the test set. With imbalanced classes that's risky: if only 10% of your data is spam, a plain split could easily end up with almost no spam in the test set.

A **stratified split** keeps the class proportions the same in both parts. Write `stratified_split(samples, test_ratio)`, where `samples` is a list of `(features, label)` pairs:

- Handle each label separately. For a label with `n` samples, `round(n * test_ratio)` of them go to the test set, namely the **last** ones of that label.
- Return `(train, test)`, with both lists keeping the samples in their original order.

```python
samples = [(1, "a"), (2, "b"), (3, "a"), (4, "a"), (5, "b"), (6, "a")]
stratified_split(samples, 0.5)
# ([(1, "a"), (2, "b"), (3, "a")], [(4, "a"), (5, "b"), (6, "a")])
```

Label `"a"` has 4 samples, so the last 2 go to test. Label `"b"` has 2, so the last 1 goes to test.

In [ ]:
def stratified_split(samples, test_ratio):
    # your code here
    pass

In [ ]:
check_stratified_split(stratified_split)

### Exercise 11: `most_similar`

The **Jaccard similarity** of two sets is the size of their intersection divided by the size of their union. It's 1.0 for identical sets and 0.0 for sets with nothing in common.

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

Write `most_similar(query, documents, k)`:

- `query` is a set of words.
- `documents` is a dict mapping a document name to its set of words.
- Return a list of the names of the `k` documents most similar to `query`, most similar first. Break ties alphabetically by name.
- If the union of two sets is empty, treat their similarity as `0.0`.

```python
docs = {"d1": {"cat", "sat", "mat"}, "d2": {"dog", "sat", "log"}, "d3": {"cat", "mat", "hat"}}
most_similar({"cat", "mat"}, docs, 2)   # ["d1", "d3"]
```

This is a very basic version of a search engine or a retrieval step: find the stored documents closest to a query.

In [ ]:
def most_similar(query, documents, k):
    # your code here
    pass

In [ ]:
check_most_similar(most_similar)

### Exercise 12: `bigram_model`

A **bigram model** predicts the next word from the current one, based on counts of which word followed which in some training text. It's the simplest possible language model.

Write `bigram_model(sentences)` that takes a list of sentences and returns a nested dictionary: `model[word][next_word]` is the number of times `next_word` came right after `word`.

- Lowercase each sentence and split it on whitespace.
- Only count pairs inside a sentence. The last word of one sentence and the first word of the next aren't a pair.
- Words that are never followed by anything shouldn't appear as top-level keys.

```python
bigram_model(["the cat sat", "the cat ran"])
# {"the": {"cat": 2}, "cat": {"sat": 1, "ran": 1}}
```

In [ ]:
def bigram_model(sentences):
    # your code here
    pass

In [ ]:
check_bigram_model(bigram_model)

## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then try the mini project in `04_mini_project.ipynb`, where you'll build a spam filter using dictionaries, sets and counters.